# ATAM-FET — Reproducible Notebook (Colab)

Runs **top to bottom** and reproduces **every number and figure** in the paper:
ablation (baseline -> +ATAM -> +TFG -> +UCH = full ATAM-FET), full-model test metrics + per-label
table, training-curve figure, confusion-matrix figure, ATAM attention heat-map, external validation
(Jigsaw Unintended Bias), efficiency benchmark, and the architecture figures — plus paste-ready
`results/latex_tables.tex`.

**Federated learning is future work** (`do_federated=False`).

**Run order:** cells top->bottom. Cell 5 first does a `quick=True` smoke test (minutes); once it
passes, switch to `quick=False` for the real run. Nothing is hard-coded — every paper number comes
from these cells.


In [14]:
import zipfile
import os

src = "/kaggle/input/competitions/jigsaw-toxic-comment-classification-challenge"
dst = "data/raw"

os.makedirs(dst, exist_ok=True)

for f in os.listdir(src):
    if f.endswith(".zip"):
        with zipfile.ZipFile(os.path.join(src, f), "r") as z:
            z.extractall(dst)

print(os.listdir(dst))

['sample_submission.csv.zip', 'sample_submission.csv', 'train.csv', 'train.csv.zip', 'test_labels.csv.zip', 'test.csv', 'test.csv.zip', 'test_labels.csv']


In [15]:
import os
import shutil
import pandas as pd
from pathlib import Path

os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/external", exist_ok=True)

# Correct paths
src = "/kaggle/input/competitions/jigsaw-toxic-comment-classification-challenge"
src2 = "/kaggle/input/competitions/jigsaw-unintended-bias-in-toxicity-classification"

# Copy all files from the primary competition
for f in Path(src).glob("*"):
    if f.is_file():
        shutil.copy(f, "data/raw")

# Copy train.csv from the external competition
shutil.copy(
    f"{src2}/train.csv",
    "data/external/train.csv"
)

# Create external evaluation sample
ub = pd.read_csv(
    "data/external/train.csv",
    usecols=["comment_text", "target"]
)

ub.sample(
    n=min(30000, len(ub)),
    random_state=42
).to_csv(
    "data/external/unintended_bias_test.csv",
    index=False
)

EXTERNAL_CSV = "/kaggle/input/competitions/jigsaw-unintended-bias-in-toxicity-classification/train.csv"

DATA_DIR = "/kaggle/input/competitions/jigsaw-toxic-comment-classification-challenge"
OUT_DIR = "results"

print("DATA_DIR =", DATA_DIR)
print("EXTERNAL_CSV =", EXTERNAL_CSV)

DATA_DIR = /kaggle/input/competitions/jigsaw-toxic-comment-classification-challenge
EXTERNAL_CSV = /kaggle/input/competitions/jigsaw-unintended-bias-in-toxicity-classification/train.csv


## 3) The ATAM-FET engine (full library, self-contained)

In [16]:
# -*- coding: utf-8 -*-
"""
ATAM-FET : Adaptive Toxic Attention Framework for Federated, Edge-based Toxicity detection
=========================================================================================
Complete, runnable research engine that produces EVERY number the paper reports:

  * Centralized training  (DistilBERT baseline  ->  +ATAM  ->  +TFG  ->  +UCH = full ATAM-FET)
  * Mandatory ablation study (one component added at a time)
  * External validation on the Jigsaw Unintended Bias dataset (binary toxic / non-toxic)
  * Federated learning (FedAvg over LoRA adapters; IID and non-IID Dirichlet partitions)
  * Explainability (ATAM token-attention heat-maps + top toxic tokens)
  * Efficiency benchmark (parameters, size MB, CPU/GPU latency)
  * Emits results as JSON  AND  ready-to-paste LaTeX tables.

The three NOVEL modules (toggled by flags so the ablation is a real experiment):
  ATAM  - Adaptive Toxic Attention Module    : learnable token-level toxicity attention -> z
  TFG   - Toxicity Feature-Gating module     : learned gate fusing [CLS] context with z
  UCH   - Uncertainty-aware Calibrated Head  : dual logit+uncertainty head + per-label
                                               threshold calibration (fixes fixed-0.5 + imbalance)

Author: Saida Binte Alam, Najmus Saquib Aurko, Anindita Roy (AIUB)
Reproducibility: seed=42 everywhere.  Tested against transformers>=4.38, torch>=2.1.
"""

from __future__ import annotations
import os, gc, re, json, math, time, random, warnings
from pathlib import Path
from dataclasses import dataclass, field, asdict
from typing import Optional, List, Dict, Tuple, Callable

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, hamming_loss, confusion_matrix)

warnings.filterwarnings("ignore")

LABEL_COLS = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]


# ============================================================================ #
#  0. Config & reproducibility                                                 #
# ============================================================================ #
@dataclass
class Config:
    model_name: str = "distilbert-base-uncased"
    max_seq_length: int = 256
    batch_size: int = 32
    eval_batch_size: int = 64
    epochs: int = 6
    learning_rate: float = 2e-5
    weight_decay: float = 0.01
    warmup_steps: int = 500
    gradient_clip: float = 1.0
    dropout: float = 0.1
    num_labels: int = 6
    seed: int = 42
    # module switches (drive the ablation)
    use_atam: bool = True
    use_tfg: bool = True
    use_uch: bool = True
    # federated
    fl_num_clients: int = 10
    fl_rounds: int = 25
    fl_local_epochs: int = 1
    fl_lora_r: int = 8
    fl_lora_alpha: float = 16.0
    fl_dirichlet_alpha: float = 0.5
    # data paths (overridden by the notebook for Kaggle / Colab)
    data_dir: str = "data/raw"                       # expects train.csv (Jigsaw toxic)
    external_csv: str = "data/external/unintended_bias_test.csv"
    out_dir: str = "results"

    def device(self) -> torch.device:
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def set_seed(seed: int = 42) -> None:
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)


# ============================================================================ #
#  1. Data loading, cleaning, splitting, tokenization                          #
# ============================================================================ #
def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return "[no_text]"
    text = re.sub(r"https?://\S+|www\.\S+", "", text, flags=re.IGNORECASE)
    text = re.sub(r"@\S+", "", text, flags=re.IGNORECASE)
    try:
        import emoji
        text = emoji.demojize(text)
    except Exception:
        pass
    text = text.lower()
    text = re.sub(r"\s+", " ", text).strip()
    return text if text else "[no_text]"


def load_jigsaw(cfg: Config) -> pd.DataFrame:
    """Load + clean the primary Jigsaw multi-label training corpus."""
    df = pd.read_csv(Path(cfg.data_dir) / "train.csv")
    df["clean_text"] = df["comment_text"].apply(clean_text)
    before = len(df)
    df = df.drop_duplicates(subset=["clean_text"], keep="first").reset_index(drop=True)
    print(f"[data] raw={before:,}  after dedupe={len(df):,}  (removed {before-len(df)})")
    for c in LABEL_COLS:
        print(f"       {c:<16s} {int(df[c].sum()):6d} positive ({100*df[c].mean():.2f}%)")
    return df


def stratified_split(df: pd.DataFrame, seed: int = 42):
    from sklearn.model_selection import train_test_split
    strat = (df[LABEL_COLS].sum(axis=1) > 0).astype(int)
    tr, tmp = train_test_split(df, test_size=0.2, random_state=seed, stratify=strat)
    strat2 = (tmp[LABEL_COLS].sum(axis=1) > 0).astype(int)
    va, te = train_test_split(tmp, test_size=0.5, random_state=seed, stratify=strat2)
    print(f"[split] train={len(tr):,}  val={len(va):,}  test={len(te):,}")
    return tr.reset_index(drop=True), va.reset_index(drop=True), te.reset_index(drop=True)


def load_external_unintended_bias(cfg: Config) -> pd.DataFrame:
    """
    Jigsaw Unintended Bias -> binary external validation set.
    Expects a CSV with a 'comment_text' column and a continuous 'toxicity'
    (or 'target') column in [0,1]; label = 1 if score >= 0.5.
    """
    p = Path(cfg.external_csv)
    if not p.exists():
        print(f"[external] file not found: {p}  (skip external validation)")
        return pd.DataFrame()
    df = pd.read_csv(p)
    score_col = "target" if "target" in df.columns else ("toxicity" if "toxicity" in df.columns else None)
    if score_col is None:
        raise ValueError("external CSV needs a 'target' or 'toxicity' column")
    df["clean_text"] = df["comment_text"].apply(clean_text)
    df["y_bin"] = (df[score_col] >= 0.5).astype(int)
    print(f"[external] {len(df):,} rows  |  toxic={int(df['y_bin'].sum()):,} "
          f"({100*df['y_bin'].mean():.2f}%)")
    return df


class ToxicDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts, self.labels = list(texts), labels
        self.tok, self.max_len = tokenizer, max_len

    def __len__(self): return len(self.texts)

    def __getitem__(self, i):
        enc = self.tok(self.texts[i], padding="max_length", truncation=True,
                       max_length=self.max_len, return_tensors="pt")
        item = {"input_ids": enc["input_ids"].squeeze(0),
                "attention_mask": enc["attention_mask"].squeeze(0)}
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[i], dtype=torch.float32)
        return item


def make_loader(df, tokenizer, cfg, train=False, labels=True):
    y = df[LABEL_COLS].values if labels else None
    ds = ToxicDataset(df["clean_text"].tolist(), y, tokenizer, cfg.max_seq_length)
    return DataLoader(ds, batch_size=cfg.batch_size if train else cfg.eval_batch_size,
                      shuffle=train, num_workers=2)


def compute_pos_weight(labels: np.ndarray) -> torch.Tensor:
    pos = labels.sum(axis=0); neg = labels.shape[0] - pos
    return torch.tensor(neg / np.maximum(pos, 1), dtype=torch.float32)


# ============================================================================ #
#  2. Novel module 1 : ATAM  (Adaptive Toxic Attention Module)                 #
# ============================================================================ #
class ATAM(nn.Module):
    r"""
    Learns a toxicity-relevance weight per token and returns a toxicity-aware
    sentence vector z = sum_i alpha_i h_i, with
        e_i = v^T tanh(W h_i + b),   alpha = softmax(e)  (padding masked).
    """
    def __init__(self, hidden_size: int):
        super().__init__()
        self.W = nn.Linear(hidden_size, hidden_size, bias=True)
        self.v = nn.Linear(hidden_size, 1, bias=False)
        nn.init.xavier_uniform_(self.W.weight); nn.init.zeros_(self.W.bias)
        nn.init.xavier_uniform_(self.v.weight)

    def forward(self, hs, attention_mask):
        e = self.v(torch.tanh(self.W(hs))).squeeze(-1)          # (B, L)
        e = e.masked_fill(attention_mask == 0, -1e9)
        alpha = F.softmax(e, dim=-1)                            # (B, L)
        z = torch.bmm(alpha.unsqueeze(1), hs).squeeze(1)        # (B, H)
        return z, alpha


# ============================================================================ #
#  3. Novel module 2 : TFG  (Toxicity Feature-Gating / gated fusion)           #
# ============================================================================ #
class TFG(nn.Module):
    r"""
    Fuses the general [CLS] context (c) and ATAM's toxicity vector (z) with a
    learned, element-wise gate instead of plain concatenation:
        p = GELU(W_p [c;z] + b_p)            (toxicity feature projection)
        g = sigmoid(W_g [c;z] + b_g)         (gate)
        u = g (*) p + (1 - g) (*) c          (fused representation, size H)
    """
    def __init__(self, hidden_size: int):
        super().__init__()
        self.proj = nn.Linear(2 * hidden_size, hidden_size)
        self.gate = nn.Linear(2 * hidden_size, hidden_size)
        self.act = nn.GELU()
        for m in (self.proj, self.gate):
            nn.init.xavier_uniform_(m.weight); nn.init.zeros_(m.bias)

    def forward(self, cls_vec, z):
        cat = torch.cat([cls_vec, z], dim=-1)                  # (B, 2H)
        p = self.act(self.proj(cat))                           # (B, H)
        g = torch.sigmoid(self.gate(cat))                      # (B, H)
        return g * p + (1.0 - g) * cls_vec                     # (B, H)


# ============================================================================ #
#  4. Novel module 3 : UCH  (Uncertainty-aware Calibrated Head)                #
# ============================================================================ #
class UCH(nn.Module):
    r"""
    Dual head: per-label logit mu_l and per-label log-variance s_l (aleatoric
    uncertainty).  Trained with a heteroscedastic BCE that down-weights noisy
    samples:   L = mean( exp(-s) * BCE(mu, y) + 0.5 * s ).
    Decision thresholds tau_l are calibrated per label on the validation set
    (see calibrate_thresholds), replacing the naive fixed 0.5.
    """
    def __init__(self, in_features: int, num_labels: int):
        super().__init__()
        self.mu = nn.Linear(in_features, num_labels)
        self.logvar = nn.Linear(in_features, num_labels)
        for m in (self.mu, self.logvar):
            nn.init.normal_(m.weight, std=0.02); nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.mu(x), self.logvar(x)                      # (B,L), (B,L)


# ============================================================================ #
#  5. The full model with toggleable components                                #
# ============================================================================ #
class ATAMFET(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.encoder = AutoModel.from_pretrained(cfg.model_name)
        H = self.encoder.config.hidden_size
        self.use_atam, self.use_tfg, self.use_uch = cfg.use_atam, cfg.use_tfg, cfg.use_uch

        if self.use_atam:
            self.atam = ATAM(H)
        # TFG only makes sense when ATAM is present
        self.use_tfg = self.use_tfg and self.use_atam
        if self.use_tfg:
            self.tfg = TFG(H)

        # classifier input dim:
        #   no ATAM                 -> H            (just [CLS])
        #   ATAM + TFG              -> H            (gated fusion)
        #   ATAM, no TFG            -> 2H           (concat [CLS], z)
        in_dim = H if (not self.use_atam or self.use_tfg) else 2 * H
        self.dropout = nn.Dropout(cfg.dropout)
        if self.use_uch:
            self.head = UCH(in_dim, cfg.num_labels)
        else:
            self.head = nn.Linear(in_dim, cfg.num_labels)
            nn.init.normal_(self.head.weight, std=0.02); nn.init.zeros_(self.head.bias)

    def _represent(self, input_ids, attention_mask, return_alpha=False):
        hs = self.encoder(input_ids, attention_mask=attention_mask).last_hidden_state
        cls_vec = hs[:, 0]
        alpha = None
        if self.use_atam:
            z, alpha = self.atam(hs, attention_mask)
            rep = self.tfg(cls_vec, z) if self.use_tfg else torch.cat([cls_vec, z], dim=-1)
        else:
            rep = cls_vec
        rep = self.dropout(rep)
        return (rep, alpha) if return_alpha else rep

    def forward(self, input_ids, attention_mask, return_alpha=False):
        out = self._represent(input_ids, attention_mask, return_alpha=return_alpha)
        rep, alpha = out if return_alpha else (out, None)
        if self.use_uch:
            mu, logvar = self.head(rep)
            return (mu, logvar, alpha) if return_alpha else (mu, logvar)
        logits = self.head(rep)
        return (logits, alpha) if return_alpha else logits

    def logits(self, input_ids, attention_mask):
        """Unified accessor: always returns class logits (mu if UCH)."""
        out = self.forward(input_ids, attention_mask)
        return out[0] if self.use_uch else out

    def num_params(self, trainable_only=True):
        return sum(p.numel() for p in self.parameters()
                   if (p.requires_grad or not trainable_only))


# ============================================================================ #
#  6. Losses                                                                   #
# ============================================================================ #
def bce_loss_fn(pos_weight, device):
    return nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device))


def uncertainty_loss(mu, logvar, target, pos_weight):
    """Heteroscedastic BCE: exp(-s)*BCE + 0.5*s, with per-label pos_weight."""
    logvar = torch.clamp(logvar, -6.0, 6.0)
    bce = F.binary_cross_entropy_with_logits(mu, target, pos_weight=pos_weight, reduction="none")
    return (torch.exp(-logvar) * bce + 0.5 * logvar).mean()


# ============================================================================ #
#  7. Metrics                                                                  #
# ============================================================================ #
def sigmoid(x): return 1.0 / (1.0 + np.exp(-x))


def compute_metrics(logits: np.ndarray, labels: np.ndarray, thresholds=None) -> Dict:
    probs = sigmoid(logits)
    thr = np.full(labels.shape[1], 0.5) if thresholds is None else np.asarray(thresholds)
    preds = (probs >= thr[None, :]).astype(np.float32)
    aucs = []
    for i in range(labels.shape[1]):
        try: aucs.append(roc_auc_score(labels[:, i], probs[:, i]))
        except ValueError: aucs.append(float("nan"))
    return {
        "subset_accuracy": accuracy_score(labels, preds),
        "f1_micro": f1_score(labels, preds, average="micro", zero_division=0),
        "f1_macro": f1_score(labels, preds, average="macro", zero_division=0),
        "precision_micro": precision_score(labels, preds, average="micro", zero_division=0),
        "recall_micro": recall_score(labels, preds, average="micro", zero_division=0),
        "precision_macro": precision_score(labels, preds, average="macro", zero_division=0),
        "recall_macro": recall_score(labels, preds, average="macro", zero_division=0),
        "roc_auc_macro": float(np.nanmean(aucs)),
        "hamming_loss": hamming_loss(labels, preds),
    }


def per_label_report(logits, labels, thresholds=None) -> pd.DataFrame:
    probs = sigmoid(logits)
    thr = np.full(labels.shape[1], 0.5) if thresholds is None else np.asarray(thresholds)
    preds = (probs >= thr[None, :]).astype(np.int32)
    rows = []
    for i, name in enumerate(LABEL_COLS):
        try: auc = roc_auc_score(labels[:, i], probs[:, i])
        except ValueError: auc = float("nan")
        rows.append(dict(label=name,
                         precision=precision_score(labels[:, i], preds[:, i], zero_division=0),
                         recall=recall_score(labels[:, i], preds[:, i], zero_division=0),
                         f1=f1_score(labels[:, i], preds[:, i], zero_division=0),
                         roc_auc=auc, support=int(labels[:, i].sum()),
                         threshold=float(thr[i])))
    return pd.DataFrame(rows)


def calibrate_thresholds(logits, labels, grid=None) -> np.ndarray:
    """Per-label threshold that maximises F1 on the (validation) set."""
    if grid is None: grid = np.linspace(0.05, 0.95, 19)
    probs = sigmoid(logits); thr = np.full(labels.shape[1], 0.5)
    for i in range(labels.shape[1]):
        best_f1, best_t = -1.0, 0.5
        for t in grid:
            f1 = f1_score(labels[:, i], (probs[:, i] >= t).astype(int), zero_division=0)
            if f1 > best_f1: best_f1, best_t = f1, t
        thr[i] = best_t
    return thr


# ============================================================================ #
#  8. Centralized training / evaluation                                        #
# ============================================================================ #
def _forward_logits(model, batch, device):
    ids = batch["input_ids"].to(device); mask = batch["attention_mask"].to(device)
    if model.use_uch:
        mu, logvar = model(ids, mask); return mu, logvar
    return model(ids, mask), None


@torch.no_grad()
def collect_logits(model, loader, device):
    model.eval(); L, Y = [], []
    for b in loader:
        mu, _ = _forward_logits(model, b, device)
        L.append(mu.cpu().numpy()); Y.append(b["labels"].numpy())
    return np.concatenate(L), np.concatenate(Y)


def train_centralized(cfg: Config, model, train_df, val_df, test_df, tokenizer,
                      tag="full", save_history=True) -> Dict:
    device = cfg.device(); model.to(device)
    tl = make_loader(train_df, tokenizer, cfg, train=True)
    vl = make_loader(val_df, tokenizer, cfg)
    pw = compute_pos_weight(train_df[LABEL_COLS].values).to(device)

    no_decay = ["bias", "LayerNorm.weight"]
    grouped = [
        {"params": [p for n, p in model.named_parameters() if p.requires_grad and not any(nd in n for nd in no_decay)],
         "weight_decay": cfg.weight_decay},
        {"params": [p for n, p in model.named_parameters() if p.requires_grad and any(nd in n for nd in no_decay)],
         "weight_decay": 0.0},
    ]
    opt = torch.optim.AdamW(grouped, lr=cfg.learning_rate)
    total = len(tl) * cfg.epochs
    sch = get_linear_schedule_with_warmup(opt, min(cfg.warmup_steps, total // 10), total)

    hist = {k: [] for k in ["train_loss", "val_loss", "val_f1_macro", "val_accuracy"]}
    best_f1, best_state = -1.0, None
    for ep in range(cfg.epochs):
        model.train(); running = 0.0
        for b in tl:
            ids = b["input_ids"].to(device); mask = b["attention_mask"].to(device)
            y = b["labels"].to(device)
            if model.use_uch:
                mu, logvar = model(ids, mask); loss = uncertainty_loss(mu, logvar, y, pw)
            else:
                loss = F.binary_cross_entropy_with_logits(model(ids, mask), y, pos_weight=pw)
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), cfg.gradient_clip)
            opt.step(); sch.step(); running += loss.item()
        vlog, vy = collect_logits(model, vl, device)
        vm = compute_metrics(vlog, vy)
        hist["train_loss"].append(running / len(tl))
        hist["val_loss"].append(float("nan")); hist["val_f1_macro"].append(vm["f1_macro"])
        hist["val_accuracy"].append(vm["subset_accuracy"])
        print(f"[{tag}] epoch {ep+1}/{cfg.epochs}  train_loss={running/len(tl):.4f} "
              f"val_f1_macro={vm['f1_macro']:.4f} val_acc={vm['subset_accuracy']:.4f}")
        if vm["f1_macro"] > best_f1:
            best_f1 = vm["f1_macro"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    if best_state: model.load_state_dict(best_state)

    # calibrate thresholds on validation (UCH benefit; harmless otherwise)
    vlog, vy = collect_logits(model, vl, device)
    thr = calibrate_thresholds(vlog, vy) if cfg.use_uch else np.full(cfg.num_labels, 0.5)

    # test
    testl = make_loader(test_df, tokenizer, cfg)
    tlog, ty = collect_logits(model, testl, device)
    test_metrics = compute_metrics(tlog, ty, thresholds=thr)
    plr = per_label_report(tlog, ty, thresholds=thr)

    out = {"tag": tag, "best_val_f1_macro": best_f1, "thresholds": thr.tolist(),
           "test_metrics": test_metrics, "per_label": plr.to_dict(orient="records"),
           "history": hist, "num_params": model.num_params(),
           "config": {k: getattr(cfg, k) for k in ["use_atam", "use_tfg", "use_uch"]}}
    if save_history:
        _dump(cfg, f"centralized_{tag}.json", out)
    print(f"[{tag}] TEST  subacc={test_metrics['subset_accuracy']:.4f} "
          f"microF1={test_metrics['f1_micro']:.4f} macroF1={test_metrics['f1_macro']:.4f} "
          f"AUC={test_metrics['roc_auc_macro']:.4f}")
    return out, (tlog, ty, thr)


# ============================================================================ #
#  9. Ablation study  (baseline -> +ATAM -> +TFG -> +UCH)                       #
# ============================================================================ #
ABLATION_STEPS = [
    ("baseline",       dict(use_atam=False, use_tfg=False, use_uch=False)),
    ("+ATAM",          dict(use_atam=True,  use_tfg=False, use_uch=False)),
    ("+ATAM+TFG",      dict(use_atam=True,  use_tfg=True,  use_uch=False)),
    ("+ATAM+TFG+UCH",  dict(use_atam=True,  use_tfg=True,  use_uch=True)),   # full ATAM-FET
]


def run_ablation(base_cfg: Config, train_df, val_df, test_df, tokenizer) -> List[Dict]:
    results = []
    for name, flags in ABLATION_STEPS:
        set_seed(base_cfg.seed)
        cfg = Config(**{**asdict(base_cfg), **flags})
        print(f"\n===== ABLATION: {name}  {flags} =====")
        model = ATAMFET(cfg)
        res, _ = train_centralized(cfg, model, train_df, val_df, test_df, tokenizer, tag=name)
        res["step"] = name
        results.append(res)
        del model; gc.collect(); torch.cuda.empty_cache()
    _dump(base_cfg, "ablation.json", {"steps": results})
    return results


# ============================================================================ #
# 10. External validation (Jigsaw Unintended Bias, binary toxic/non-toxic)     #
# ============================================================================ #
@torch.no_grad()
def external_validation(cfg: Config, model, ext_df, tokenizer,
                        toxic_index=0) -> Dict:
    """
    Binary external test: use the model's 'toxic' head probability (and the
    'any-label' variant) as the toxicity score against the binary ground truth.
    """
    if ext_df is None or len(ext_df) == 0:
        print("[external] skipped (no data)"); return {}
    device = cfg.device(); model.to(device).eval()
    loader = make_loader(ext_df, tokenizer, cfg, labels=False)
    scores = []
    for b in loader:
        mu = model.logits(b["input_ids"].to(device), b["attention_mask"].to(device))
        scores.append(sigmoid(mu.cpu().numpy()))
    P = np.concatenate(scores)                       # (N, 6) probabilities
    y = ext_df["y_bin"].values
    def _bin(score):
        pred = (score >= 0.5).astype(int)
        try: auc = roc_auc_score(y, score)
        except ValueError: auc = float("nan")
        return dict(accuracy=accuracy_score(y, pred),
                    precision=precision_score(y, pred, zero_division=0),
                    recall=recall_score(y, pred, zero_division=0),
                    f1=f1_score(y, pred, zero_division=0), roc_auc=auc)
    res = {"toxic_head": _bin(P[:, toxic_index]),
           "any_label_max": _bin(P.max(axis=1))}
    _dump(cfg, "external_unintended_bias.json", res)
    print(f"[external] toxic-head  acc={res['toxic_head']['accuracy']:.4f} "
          f"F1={res['toxic_head']['f1']:.4f} AUC={res['toxic_head']['roc_auc']:.4f}")
    return res


# ============================================================================ #
# 11. Federated learning : LoRA + FedAvg, IID and non-IID (Dirichlet)          #
# ============================================================================ #
class LoRALinear(nn.Module):
    def __init__(self, base: nn.Linear, r=8, alpha=16.0):
        super().__init__()
        self.base = base
        self.base.weight.requires_grad_(False)
        if self.base.bias is not None: self.base.bias.requires_grad_(False)
        self.A = nn.Parameter(torch.randn(r, base.in_features) * 0.02)
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        self.s = alpha / r

    def forward(self, x):
        return self.base(x) + ((x @ self.A.T) @ self.B.T) * self.s


def inject_lora(model, targets=("q_lin", "v_lin"), r=8, alpha=16.0) -> int:
    n = 0
    for name, module in list(model.named_modules()):
        for t in targets:
            if isinstance(module, nn.Linear) and name.endswith(t):
                parent = model
                *path, last = name.split(".")
                for p in path:
                    parent = parent[int(p)] if p.isdigit() else getattr(parent, p)
                setattr(parent, last, LoRALinear(module, r, alpha)); n += 1
    return n


def freeze_encoder(model):
    for p in model.encoder.parameters(): p.requires_grad_(False)


def trainable_state(model) -> List[np.ndarray]:
    return [p.detach().cpu().numpy() for p in model.parameters() if p.requires_grad]


def load_trainable(model, arrays):
    for p, a in zip([p for p in model.parameters() if p.requires_grad], arrays):
        p.data.copy_(torch.tensor(a, device=p.device))


def partition_indices(df, num_clients, mode="iid", alpha=0.5, seed=42):
    rng = np.random.RandomState(seed); n = len(df)
    if mode == "iid":
        idx = rng.permutation(n)
        return [idx[i::num_clients] for i in range(num_clients)]
    # non-IID Dirichlet over the toxic/non-toxic flag
    flag = (df[LABEL_COLS].sum(axis=1) > 0).astype(int).values
    parts = [[] for _ in range(num_clients)]
    for c in [0, 1]:
        c_idx = np.where(flag == c)[0]; rng.shuffle(c_idx)
        prop = rng.dirichlet([alpha] * num_clients)
        cuts = (np.cumsum(prop) * len(c_idx)).astype(int)[:-1]
        for k, chunk in enumerate(np.split(c_idx, cuts)):
            parts[k].extend(chunk.tolist())
    return [np.array(p) for p in parts]


def run_federated(cfg: Config, train_df, val_df, test_df, tokenizer, mode="iid") -> Dict:
    device = cfg.device()
    print(f"\n===== FEDERATED ({mode}) : {cfg.fl_num_clients} clients x {cfg.fl_rounds} rounds =====")
    set_seed(cfg.seed)
    global_model = ATAMFET(cfg).to(device)
    freeze_encoder(global_model)
    n_lora = inject_lora(global_model, r=cfg.fl_lora_r, alpha=cfg.fl_lora_alpha)
    global_model.to(device)
    comm_params = sum(p.numel() for p in global_model.parameters() if p.requires_grad)
    print(f"[fl] injected {n_lora} LoRA adapters | communicated params/round/client = {comm_params:,}")

    parts = partition_indices(train_df, cfg.fl_num_clients, mode, cfg.fl_dirichlet_alpha, cfg.seed)
    client_loaders = [make_loader(train_df.iloc[p].reset_index(drop=True), tokenizer, cfg, train=True)
                      for p in parts]
    vl = make_loader(val_df, tokenizer, cfg)
    pw = compute_pos_weight(train_df[LABEL_COLS].values).to(device)

    global_state = trainable_state(global_model)
    history = {"round": [], "val_f1_macro": [], "val_accuracy": []}
    best_f1, best_state = -1.0, None

    for rnd in range(1, cfg.fl_rounds + 1):
        updates, sizes = [], []
        for cid, loader in enumerate(client_loaders):
            load_trainable(global_model, global_state)
            opt = torch.optim.AdamW([p for p in global_model.parameters() if p.requires_grad],
                                    lr=cfg.learning_rate * 5)  # LoRA tolerates higher LR
            global_model.train()
            for _ in range(cfg.fl_local_epochs):
                for b in loader:
                    ids = b["input_ids"].to(device); mask = b["attention_mask"].to(device)
                    y = b["labels"].to(device)
                    if global_model.use_uch:
                        mu, lv = global_model(ids, mask); loss = uncertainty_loss(mu, lv, y, pw)
                    else:
                        loss = F.binary_cross_entropy_with_logits(global_model(ids, mask), y, pos_weight=pw)
                    opt.zero_grad(); loss.backward()
                    nn.utils.clip_grad_norm_(global_model.parameters(), cfg.gradient_clip)
                    opt.step()
            updates.append(trainable_state(global_model)); sizes.append(len(loader.dataset))
        # FedAvg (sample-weighted)
        tot = float(sum(sizes))
        global_state = [sum(sizes[c] / tot * updates[c][i] for c in range(len(updates)))
                        for i in range(len(updates[0]))]
        load_trainable(global_model, global_state)
        vlog, vy = collect_logits(global_model, vl, device); vm = compute_metrics(vlog, vy)
        history["round"].append(rnd); history["val_f1_macro"].append(vm["f1_macro"])
        history["val_accuracy"].append(vm["subset_accuracy"])
        if rnd % 5 == 0 or rnd == cfg.fl_rounds:
            print(f"[fl:{mode}] round {rnd}/{cfg.fl_rounds}  val_f1_macro={vm['f1_macro']:.4f} "
                  f"val_acc={vm['subset_accuracy']:.4f}")
        if vm["f1_macro"] > best_f1:
            best_f1 = vm["f1_macro"]; best_state = [a.copy() for a in global_state]

    load_trainable(global_model, best_state)
    testl = make_loader(test_df, tokenizer, cfg)
    tlog, ty = collect_logits(global_model, testl, device)
    res = {"mode": mode, "num_clients": cfg.fl_num_clients, "rounds": cfg.fl_rounds,
           "communicated_params_per_round_per_client": comm_params,
           "best_val_f1_macro": best_f1,
           "test_metrics": compute_metrics(tlog, ty), "history": history}
    _dump(cfg, f"federated_{mode}.json", res)
    print(f"[fl:{mode}] TEST microF1={res['test_metrics']['f1_micro']:.4f} "
          f"macroF1={res['test_metrics']['f1_macro']:.4f}")
    del global_model; gc.collect(); torch.cuda.empty_cache()
    return res


# ============================================================================ #
# 12. Explainability : ATAM attention heat-map + top toxic tokens              #
# ============================================================================ #
@torch.no_grad()
def explain(cfg: Config, model, tokenizer, texts: List[str], out_png=None, topk=8) -> List[Dict]:
    if not model.use_atam:
        print("[explain] model has no ATAM; skipping"); return []
    device = cfg.device(); model.to(device).eval()
    results = []
    for text in texts:
        ct = clean_text(text)
        enc = tokenizer(ct, return_tensors="pt", truncation=True, max_length=cfg.max_seq_length)
        ids = enc["input_ids"].to(device); mask = enc["attention_mask"].to(device)
        out = model(ids, mask, return_alpha=True)
        alpha = out[-1][0].cpu().numpy()
        toks = tokenizer.convert_ids_to_tokens(enc["input_ids"][0])
        order = np.argsort(alpha[:len(toks)])[::-1]
        top = [(toks[i], float(alpha[i])) for i in order[:topk] if toks[i] not in ("[CLS]", "[SEP]", "[PAD]")]
        results.append({"text": text, "top_tokens": top})
        print(f"[explain] '{text[:50]}...'  ->  " + ", ".join(f"{t}:{w:.3f}" for t, w in top[:5]))
    if out_png:
        _plot_attention(results, tokenizer, cfg, model, texts, out_png)
    _dump(cfg, "explainability.json", {"samples": results})
    return results


def _plot_attention(results, tokenizer, cfg, model, texts, out_png):
    try:
        import matplotlib.pyplot as plt
    except Exception:
        return
    device = cfg.device()
    fig, axes = plt.subplots(len(texts), 1, figsize=(12, 1.4 * len(texts) + 1))
    if len(texts) == 1: axes = [axes]
    for ax, text in zip(axes, texts):
        enc = tokenizer(clean_text(text), return_tensors="pt", truncation=True, max_length=64)
        ids = enc["input_ids"].to(device); mask = enc["attention_mask"].to(device)
        alpha = model(ids, mask, return_alpha=True)[-1][0].cpu().numpy()
        toks = tokenizer.convert_ids_to_tokens(enc["input_ids"][0]); n = len(toks)
        ax.imshow(alpha[None, :n], aspect="auto", cmap="Reds")
        ax.set_xticks(range(n)); ax.set_xticklabels(toks, rotation=45, ha="right", fontsize=7)
        ax.set_yticks([]); ax.set_title(text[:60], fontsize=8, loc="left")
    plt.tight_layout(); plt.savefig(out_png, dpi=150, bbox_inches="tight"); plt.close()
    print(f"[explain] saved heat-map -> {out_png}")


# ============================================================================ #
# 12b. Result figures reproduced by the notebook (curves + confusion)          #
# ============================================================================ #
def plot_training_curves(history: Dict, out_png: str):
    """Training/validation loss and validation macro-F1 over epochs."""
    try:
        import matplotlib.pyplot as plt
    except Exception:
        return
    ep = range(1, len(history["train_loss"]) + 1)
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(ep, history["train_loss"], "b-o", label="train loss")
    if any(not (isinstance(v, float) and v != v) for v in history["val_loss"]):
        ax[0].plot(ep, history["val_loss"], "r-o", label="val loss")
    ax[0].set_xlabel("epoch"); ax[0].set_ylabel("loss"); ax[0].set_title("Loss")
    ax[0].legend(); ax[0].grid(alpha=0.3)
    ax[1].plot(ep, history["val_f1_macro"], "g-o", label="val F1-macro")
    ax[1].set_xlabel("epoch"); ax[1].set_ylabel("F1-macro"); ax[1].set_title("Validation F1-macro")
    ax[1].legend(); ax[1].grid(alpha=0.3)
    plt.tight_layout(); plt.savefig(out_png, dpi=150, bbox_inches="tight"); plt.close()
    print(f"[fig] training curves -> {out_png}")


def plot_confusion(logits, labels, thresholds, out_png: str):
    """2x3 grid of per-label confusion matrices with P/R/F1 annotations."""
    try:
        import matplotlib.pyplot as plt
        import seaborn as sns
    except Exception:
        return
    probs = sigmoid(logits); thr = np.asarray(thresholds)
    preds = (probs >= thr[None, :]).astype(int); y = labels.astype(int)
    fig, axes = plt.subplots(2, 3, figsize=(12, 8)); axes = axes.flatten()
    for i, name in enumerate(LABEL_COLS):
        cm = confusion_matrix(y[:, i], preds[:, i], labels=[0, 1])
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[i],
                    xticklabels=["Neg", "Pos"], yticklabels=["Neg", "Pos"], cbar=False)
        tn, fp, fn, tp = cm.ravel()
        p = tp / (tp + fp) if (tp + fp) else 0.0
        r = tp / (tp + fn) if (tp + fn) else 0.0
        f = 2 * p * r / (p + r) if (p + r) else 0.0
        axes[i].set_title(f"{name}\nP={p:.3f} R={r:.3f} F1={f:.3f}", fontsize=9)
    plt.tight_layout(); plt.savefig(out_png, dpi=150, bbox_inches="tight"); plt.close()
    print(f"[fig] confusion matrices -> {out_png}")


# ============================================================================ #
# 13. Efficiency benchmark                                                     #
# ============================================================================ #
@torch.no_grad()
def benchmark(cfg: Config, model, tokenizer, n_iters=50) -> Dict:
    device = cfg.device(); model.to(device).eval()
    dummy = tokenizer("this is a benchmark sentence for latency measurement",
                      padding="max_length", truncation=True,
                      max_length=cfg.max_seq_length, return_tensors="pt")
    ids = dummy["input_ids"].to(device); mask = dummy["attention_mask"].to(device)
    for _ in range(5): model.logits(ids, mask)              # warmup
    if device.type == "cuda": torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(n_iters): model.logits(ids, mask)
    if device.type == "cuda": torch.cuda.synchronize()
    latency_ms = (time.time() - t0) / n_iters * 1000
    size_mb = sum(p.numel() * p.element_size() for p in model.parameters()) / 1e6
    res = {"params_total": model.num_params(trainable_only=False),
           "params_trainable": model.num_params(True),
           "size_fp32_mb": round(size_mb, 2),
           "latency_ms_bs1": round(latency_ms, 2),
           "device": device.type}
    _dump(cfg, "benchmark.json", res)
    print(f"[bench] params={res['params_total']:,} size={res['size_fp32_mb']}MB "
          f"latency={res['latency_ms_bs1']}ms/{device.type}")
    return res


# ============================================================================ #
# 14. Result serialization + LaTeX table emission                             #
# ============================================================================ #
def _dump(cfg: Config, name: str, obj: Dict):
    Path(cfg.out_dir).mkdir(parents=True, exist_ok=True)
    with open(Path(cfg.out_dir) / name, "w") as f:
        json.dump(obj, f, indent=2, default=float)


def emit_latex_tables(cfg: Config):
    """Read the produced JSON files and write paste-ready LaTeX table bodies."""
    out = Path(cfg.out_dir); lines = []

    def r(name):
        p = out / name
        return json.load(open(p)) if p.exists() else None

    def tex_esc(s):                      # escape LaTeX-special underscores
        return str(s).replace("_", "\\_")

    abl = r("ablation.json")
    if abl:
        lines.append("% ===== Ablation table =====")
        lines.append("\\begin{tabular}{@{}lccccc@{}}\\toprule")
        lines.append("Configuration & Subset Acc & micro-F1 & macro-F1 & ROC-AUC & Params (M)\\\\ \\midrule")
        for s in abl["steps"]:
            m = s["test_metrics"]; step = tex_esc(s["step"])
            lines.append(f"{step} & {m['subset_accuracy']:.3f} & "
                         f"{m['f1_micro']:.3f} & {m['f1_macro']:.3f} & {m['roc_auc_macro']:.3f} & "
                         f"{s['num_params']/1e6:.2f}\\\\")
        lines.append("\\bottomrule\\end{tabular}\n")

    full = r("centralized_+ATAM+TFG+UCH.json") or r("centralized_full.json")
    if full:
        m = full["test_metrics"]
        lines.append("% ===== Full-model test metrics =====")
        for k in ["subset_accuracy", "hamming_loss", "roc_auc_macro", "f1_micro",
                  "precision_micro", "recall_micro", "f1_macro", "precision_macro", "recall_macro"]:
            lines.append(f"{k.replace('_',' ')} & {m[k]:.3f}\\\\")
        lines.append("")
        lines.append("% ===== Per-label table =====")
        lines.append("\\begin{tabular}{@{}lcccc@{}}\\toprule")
        lines.append("Label & Precision & Recall & F1 & Support\\\\ \\midrule")
        for row in full["per_label"]:
            lbl = tex_esc(row["label"])
            lines.append(f"{lbl} & {row['precision']:.3f} & "
                         f"{row['recall']:.3f} & {row['f1']:.3f} & {int(row['support'])}\\\\")
        lines.append("\\bottomrule\\end{tabular}\n")

    for mode in ["iid", "non_iid", "dirichlet"]:
        fl = r(f"federated_{mode}.json")
        if fl:
            m = fl["test_metrics"]
            lines.append(f"% ===== Federated ({mode}) =====")
            lines.append(f"FedAvg+LoRA ({mode}) & {m['subset_accuracy']:.3f} & {m['f1_micro']:.3f} & "
                         f"{m['f1_macro']:.3f} & {fl['communicated_params_per_round_per_client']/1e6:.3f}M/round\\\\")

    ext = r("external_unintended_bias.json")
    if ext:
        lines.append("% ===== External validation (Jigsaw Unintended Bias) =====")
        for k, v in ext.items():
            lines.append(f"{k.replace('_',' ')} & {v['accuracy']:.3f} & {v['f1']:.3f} & {v['roc_auc']:.3f}\\\\")

    txt = "\n".join(lines)
    (out / "latex_tables.tex").write_text(txt, encoding="utf-8")
    print(f"[latex] wrote {out/'latex_tables.tex'}  ({len(lines)} lines)")
    return txt


## 4) Orchestrator

In [17]:
# -*- coding: utf-8 -*-
"""
ATAM-FET : one-shot orchestrator (centralized; FL is future work).

Reproduces EVERY number and figure the paper reports:
  * ablation  baseline -> +ATAM -> +TFG -> +UCH   (the last = full ATAM-FET)
  * full-model test metrics + per-label table
  * training-curve figure, confusion-matrix figure, ATAM attention heat-map
  * external validation (Jigsaw Unintended Bias)
  * efficiency benchmark
  * results/latex_tables.tex  (paste-ready)  and  results/SUMMARY.json

The full model is REUSED from the last ablation step (no redundant retrain).
Federated learning is available (do_federated=True) but OFF by default — it is
reported as future work in the paper.

Usage in the notebook:
    main(Config(data_dir=..., external_csv=..., out_dir=...), quick=True)   # smoke test
    main(Config(...), quick=False)                                          # real run
"""
import gc, json
from dataclasses import asdict
import torch
from transformers import AutoTokenizer

# NOTE: in the notebook, the engine (atam_fet.py) is inlined above this cell,
# so all names below are already in the global namespace. As a standalone
# script, uncomment the import:
# from atam_fet import *   # noqa

DEMO_TEXTS = [
    "you are so stupid and worthless",
    "thanks for the help, i really appreciate it",
    "i will find you and hurt you",
    "great edit, well sourced and clearly written",
    "people like you should not exist",
]


def main(cfg, quick=False, do_ablation=True, do_external=True,
         do_explain=True, do_bench=True, do_federated=False):
    set_seed(cfg.seed)
    tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)

    # ---------------- data ----------------
    df = load_jigsaw(cfg)
    if quick:
        df = df.sample(n=min(4000, len(df)), random_state=cfg.seed).reset_index(drop=True)
        cfg.epochs = 1
    train_df, val_df, test_df = stratified_split(df, cfg.seed)
    ext_df = load_external_unintended_bias(cfg) if do_external else None
    if quick and ext_df is not None and len(ext_df) > 2000:
        ext_df = ext_df.sample(n=2000, random_state=cfg.seed).reset_index(drop=True)

    summary = {}
    full_model = full_cfg = full_res = full_outputs = None

    # ---------------- 1. ablation (keep the full model) ----------------
    if do_ablation:
        abl = []
        for name, flags in ABLATION_STEPS:
            set_seed(cfg.seed)
            acfg = Config(**{**asdict(cfg), **flags})
            print(f"\n===== ABLATION: {name}  {flags} =====")
            model = ATAMFET(acfg)
            res, outs = train_centralized(acfg, model, train_df, val_df, test_df, tokenizer, tag=name)
            res["step"] = name
            abl.append(res)
            if name == "+ATAM+TFG+UCH":      # this IS the full ATAM-FET model
                full_model, full_cfg, full_res, full_outputs = model, acfg, res, outs
            else:
                del model; gc.collect(); torch.cuda.empty_cache()
        _dump(cfg, "ablation.json", {"steps": abl})
        summary["ablation"] = [{"step": s["step"], **s["test_metrics"], "params": s["num_params"]} for s in abl]

    # ---------------- 2. ensure we have a full model ----------------
    if full_model is None:
        set_seed(cfg.seed)
        full_cfg = Config(**{**asdict(cfg), "use_atam": True, "use_tfg": True, "use_uch": True})
        full_model = ATAMFET(full_cfg)
        full_res, full_outputs = train_centralized(full_cfg, full_model, train_df, val_df, test_df,
                                                   tokenizer, tag="full")
    summary["full_test"] = full_res["test_metrics"]
    summary["full_per_label"] = full_res["per_label"]

    # ---------------- 3. result FIGURES (reproduced live) ----------------
    tlog, ty, thr = full_outputs
    plot_training_curves(full_res["history"], f"{cfg.out_dir}/training_curves.png")
    plot_confusion(tlog, ty, thr, f"{cfg.out_dir}/confusion_matrices.png")

    # ---------------- 4. external validation ----------------
    if do_external and ext_df is not None and len(ext_df):
        summary["external"] = external_validation(full_cfg, full_model, ext_df, tokenizer)

    # ---------------- 5. explainability ----------------
    if do_explain:
        summary["explain"] = explain(full_cfg, full_model, tokenizer, DEMO_TEXTS,
                                     out_png=f"{cfg.out_dir}/attention_heatmap.png")

    # ---------------- 6. efficiency benchmark ----------------
    if do_bench:
        summary["benchmark_full"] = benchmark(full_cfg, full_model, tokenizer)

    del full_model; gc.collect(); torch.cuda.empty_cache()

    # ---------------- 7. (optional) federated — future work ----------------
    if do_federated:
        summary["federated_iid"] = run_federated(cfg, train_df, val_df, test_df, tokenizer, mode="iid")
        summary["federated_non_iid"] = run_federated(cfg, train_df, val_df, test_df, tokenizer, mode="non_iid")

    # ---------------- 8. emit LaTeX + summary ----------------
    emit_latex_tables(cfg)
    with open(f"{cfg.out_dir}/SUMMARY.json", "w") as f:
        json.dump(summary, f, indent=2, default=float)
    print("\n================ DONE ================")
    print(f"Results in {cfg.out_dir}/  (JSON, latex_tables.tex, training_curves.png, "
          f"confusion_matrices.png, attention_heatmap.png)")
    return summary


## 5) Run  (smoke test, then real run)

In [ ]:
# 5) Configure and run  (Federated learning is FUTURE WORK -> do_federated=False)
cfg = Config(
    data_dir="data/raw",
    external_csv=EXTERNAL_CSV,
    out_dir=OUT_DIR,
    epochs=4,           # fast real run (~3 h on a T4). Use epochs=6 for the paper-grade run.
)

# STEP 1 — smoke test on a tiny subset (a few minutes). Confirms the whole pipeline runs.
# summary = main(cfg, quick=True, do_federated=False)
# print("\nSMOKE TEST OK — review the numbers above.")

# STEP 2 — the REAL run. Comment out the smoke line above, then run this:
summary = main(cfg, quick=False, do_federated=False)
import json; print(json.dumps(summary["full_test"], indent=2))


[data] raw=159,571  after dedupe=159,289  (removed 282)
       toxic             15246 positive (9.57%)
       severe_toxic       1590 positive (1.00%)
       obscene            8420 positive (5.29%)
       threat              474 positive (0.30%)
       insult             7850 positive (4.93%)
       identity_hate      1397 positive (0.88%)
[split] train=127,431  val=15,929  test=15,929
[external] 1,804,874 rows  |  toxic=144,334 (8.00%)

===== ABLATION: baseline  {'use_atam': False, 'use_tfg': False, 'use_uch': False} =====


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[baseline] epoch 1/4  train_loss=0.4704 val_f1_macro=0.5838 val_acc=0.8689
[baseline] epoch 2/4  train_loss=0.2838 val_f1_macro=0.6111 val_acc=0.8861
[baseline] epoch 3/4  train_loss=0.1921 val_f1_macro=0.6584 val_acc=0.9039
[baseline] epoch 4/4  train_loss=0.1394 val_f1_macro=0.6672 val_acc=0.9100
[baseline] TEST  subacc=0.9090 microF1=0.7566 macroF1=0.6694 AUC=0.9885

===== ABLATION: +ATAM  {'use_atam': True, 'use_tfg': False, 'use_uch': False} =====


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

...

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[+ATAM+TFG] epoch 1/4  train_loss=0.4902 val_f1_macro=0.6164 val_acc=0.8878
[+ATAM+TFG] epoch 2/4  train_loss=0.3007 val_f1_macro=0.6283 val_acc=0.8947
[+ATAM+TFG] epoch 3/4  train_loss=0.2140 val_f1_macro=0.6601 val_acc=0.9041
[+ATAM+TFG] epoch 4/4  train_loss=0.1479 val_f1_macro=0.6713 val_acc=0.9095
[+ATAM+TFG] TEST  subacc=0.9099 microF1=0.7603 macroF1=0.6835 AUC=0.9906

===== ABLATION: +ATAM+TFG+UCH  {'use_atam': True, 'use_tfg': True, 'use_uch': True} =====


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[+ATAM+TFG+UCH] epoch 1/4  train_loss=211.6829 val_f1_macro=0.5908 val_acc=0.8913


## 6) Reproduce the architecture figures (Fig. 1-4)

In [ ]:
# -*- coding: utf-8 -*-
"""Publication-quality vector figures for the ATAM-FET paper (PDF + PNG).

Coordinate system: x in [0,100], y in [0, ymax]; the figure height is chosen so
the aspect is ~1:1 (no distortion of rounded boxes). All content is designed to
sit strictly inside [0, ymax].
"""
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
import os

HERE = "results/figures"; os.makedirs(HERE, exist_ok=True)
C = dict(blue="#DCE9F7", green="#DDEFDD", red="#F8DEDE", yellow="#FCEFD0",
         purple="#E9E0F3", grey="#EDEFF2", teal="#D8EEEA")
STROKE = "#33404F"


def new_ax(width, ymax):
    fig, ax = plt.subplots(figsize=(width, width * ymax / 100.0))
    ax.set_xlim(0, 100); ax.set_ylim(0, ymax); ax.axis("off")
    ax.set_aspect("auto")
    return fig, ax


def box(ax, x, y, w, h, text, fill="#FFFFFF", fs=8, bold=False, mono=False):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.02,rounding_size=0.5",
                                linewidth=1.1, edgecolor=STROKE, facecolor=fill, zorder=2))
    ax.text(x + w / 2, y + h / 2, text, ha="center", va="center", fontsize=fs,
            fontweight="bold" if bold else "normal", zorder=3,
            family="monospace" if mono else "sans-serif", color="#111")


def group(ax, x, y, w, h, title, color):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.02,rounding_size=0.8",
                                linewidth=1.3, edgecolor=color, facecolor="none",
                                linestyle=(0, (5, 3)), zorder=1))
    ax.text(x + 0.6, y + h - 0.4, title, ha="left", va="top", fontsize=9.5,
            fontweight="bold", color=color, zorder=3)


def arrow(ax, p1, p2, color=STROKE, rad=0.0):
    ax.add_patch(FancyArrowPatch(p1, p2, arrowstyle="-|>", mutation_scale=12, lw=1.2,
                                 color=color, connectionstyle=f"arc3,rad={rad}", zorder=2))


def caption(ax, ymax, text):
    ax.text(50, 0.4, text, ha="center", va="bottom", fontsize=8.2, color="#333")


def save(fig, name):
    for ext in ("pdf", "png"):
        fig.savefig(os.path.join(HERE, f"{name}.{ext}"), dpi=200, bbox_inches="tight", transparent=True)
    plt.close(fig); print("wrote", name)


# ============================ Figure 1 : pipeline ============================ #
def fig1():
    ymax = 62
    fig, ax = new_ax(14, ymax)
    # groups
    group(ax, 1, 30, 17, 28, "Input & Embedding", "#2E6CA4")
    group(ax, 20, 28, 20, 30, "DistilBERT Encoder", "#1E7E45")
    group(ax, 43, 20, 27, 38, "ATAM-FET  (proposed novel modules)", "#C0392B")
    group(ax, 20, 3, 76, 12, "Training & validation protocol", "#7A5AA6")
    # input
    box(ax, 2.5, 50, 14, 5, "Input text", C["blue"])
    box(ax, 2.5, 43, 14, 5, "WordPiece tokenizer\n(max len 256)", C["blue"], fs=7.5)
    box(ax, 2.5, 35.5, 14, 5, "Token + Positional\nemb. (d=768)", C["blue"], fs=7.5)
    # encoder
    box(ax, 22, 50, 16, 5, "Multi-Head Self-Attn (12h)", C["green"], fs=7)
    box(ax, 22, 44, 16, 4.5, "Add & LayerNorm", C["green"], fs=7.5)
    box(ax, 22, 38, 16, 4.5, "Feed-Forward Net", C["green"], fs=7.5)
    box(ax, 22, 32, 16, 4.5, "x6 -> H = {h1..hn}", C["green"], fs=7.5, mono=True)
    # atam-fet
    box(ax, 45, 50, 23, 5, "ATAM  (Fig. 2)", C["red"], fs=8, bold=True)
    box(ax, 45, 43.5, 23, 5, "TFG  (Fig. 3)", C["red"], fs=8, bold=True)
    box(ax, 45, 37, 23, 5, "UCH  (Fig. 4)", C["red"], fs=8, bold=True)
    box(ax, 45, 22, 23, 12, "z = sum_i a_i h_i\nu = g*p + (1-g)*c\nmu, sigma^2 per label", C["red"], fs=7, mono=True)
    # output
    box(ax, 74, 47, 23, 6, "Six sigmoid outputs\n(multi-label)", C["yellow"], fs=7.5)
    box(ax, 74, 35, 23, 9, "Labels:\ntoxic, severe, obscene,\nthreat, insult, id-hate", C["purple"], fs=7.5)
    # protocol band
    box(ax, 22, 5.5, 22, 7, "Federated LoRA+FedAvg\n(future work)", C["grey"], fs=7.5)
    box(ax, 49, 5.5, 20, 7, "Centralized training\n(AdamW)", C["teal"], fs=7.5)
    box(ax, 72, 5.5, 22, 7, "External validation\n(Jigsaw Unintended Bias)", C["teal"], fs=7.5)
    # arrows (main flow at y=45)
    arrow(ax, (16.5, 45), (20, 45)); arrow(ax, (38, 45), (43, 45))
    arrow(ax, (68, 45), (74, 49)); arrow(ax, (85, 47), (85, 44))
    arrow(ax, (56.5, 50), (56.5, 48.5)); arrow(ax, (56.5, 43.5), (56.5, 42))
    caption(ax, ymax, "Figure 1.  ATAM-FET end-to-end pipeline. Top row: on-device model (input -> encoder -> "
                      "ATAM -> TFG -> UCH -> labels). Federated learning is future work.")
    save(fig, "fig1_pipeline")


# ============================ Figure 2 : ATAM =============================== #
def fig2():
    ymax = 40
    fig, ax = new_ax(13, ymax)
    group(ax, 1, 3, 98, 35, "ATAM  —  Adaptive Toxic Attention Module", "#C0392B")
    box(ax, 3, 22, 16, 10, "Token emb.\nH=(h1..hn)\n(B,L,768)", C["green"], fs=7.5, mono=True)
    box(ax, 23, 22, 15, 10, "Linear W\ntanh(Wh+b)\n(B,L,768)", C["red"], fs=7.5, mono=True)
    box(ax, 42, 22, 14, 10, "score v^T(.)\ne_i (B,L)", C["red"], fs=7.5, mono=True)
    box(ax, 60, 22, 16, 10, "mask pad +\nsoftmax\nalpha (B,L)", C["red"], fs=7.5, mono=True)
    box(ax, 80, 22, 17, 10, "z = sum a_i h_i\n(B,768)", C["yellow"], fs=7.5, mono=True, bold=True)
    for x1, x2 in [(19, 23), (38, 42), (56, 60), (76, 80)]:
        arrow(ax, (x1, 27), (x2, 27))
    ax.text(50, 14, r"$e_i=v^{\top}\tanh(Wh_i+b)\quad\alpha_i=\mathrm{softmax}(e_i)\quad z=\sum_i\alpha_i h_i$",
            ha="center", fontsize=11)
    ax.text(50, 8, "Learns a toxicity-relevance weight per token; alpha is exported for explainability heat-maps.",
            ha="center", fontsize=8, style="italic", color="#555")
    caption(ax, ymax, "Figure 2.  ATAM internals.")
    save(fig, "fig2_atam")


# ============================ Figure 3 : TFG ================================ #
def fig3():
    ymax = 46
    fig, ax = new_ax(13, ymax)
    group(ax, 1, 3, 98, 41, "TFG  —  Toxicity Feature-Gating (gated fusion)", "#C0392B")
    box(ax, 3, 29, 17, 9, "[CLS] context c\n(B,768)", C["green"], fs=7.5, mono=True)
    box(ax, 3, 13, 17, 9, "ATAM vector z\n(B,768)", C["yellow"], fs=7.5, mono=True)
    box(ax, 25, 21, 15, 9, "concat [c;z]\n(B,1536)", C["grey"], fs=7.5, mono=True)
    box(ax, 45, 29, 21, 9, "proj: GELU(Wp[.])\np (B,768)", C["red"], fs=7.5, mono=True)
    box(ax, 45, 13, 21, 9, "gate: sig(Wg[.])\ng (B,768)", C["red"], fs=7.5, mono=True)
    box(ax, 72, 21, 24, 9, "u = g*p + (1-g)*c\n(B,768)", C["yellow"], fs=7.5, mono=True, bold=True)
    arrow(ax, (20, 33), (25, 27)); arrow(ax, (20, 17), (25, 24))
    arrow(ax, (40, 26), (45, 33)); arrow(ax, (40, 24), (45, 17))
    arrow(ax, (66, 33), (72, 27)); arrow(ax, (66, 17), (72, 24))
    ax.text(50, 8, r"$p=\mathrm{GELU}(W_p[c;z]+b_p),\ \ g=\sigma(W_g[c;z]+b_g),\ \ u=g\odot p+(1-g)\odot c$",
            ha="center", fontsize=10)
    caption(ax, ymax, "Figure 3.  TFG internals: a learned gate fuses general [CLS] context with ATAM's toxic evidence.")
    save(fig, "fig3_tfg")


# ============================ Figure 4 : UCH ================================ #
def fig4():
    ymax = 46
    fig, ax = new_ax(13, ymax)
    group(ax, 1, 3, 98, 41, "UCH  —  Uncertainty-aware Calibrated Head", "#C0392B")
    box(ax, 3, 22, 16, 9, "fused u\n(B,768)", C["yellow"], fs=7.5, mono=True)
    box(ax, 25, 30, 21, 8, "logit head\nmu (B,6)", C["red"], fs=7.5, mono=True)
    box(ax, 25, 14, 21, 8, "uncertainty head\nlog var (B,6)", C["red"], fs=7.5, mono=True)
    box(ax, 52, 21, 22, 10, "heteroscedastic BCE\nexp(-s)BCE + 0.5 s", C["grey"], fs=7.5, mono=True)
    box(ax, 79, 21, 18, 10, "per-label tau_l\n(calibrated,\nreplaces 0.5)", C["purple"], fs=7.5, mono=True)
    arrow(ax, (19, 28), (25, 34)); arrow(ax, (19, 24), (25, 18))
    arrow(ax, (46, 34), (52, 28)); arrow(ax, (46, 18), (52, 24))
    arrow(ax, (74, 26), (79, 26))
    ax.text(50, 9, r"$\mathcal{L}=\mathrm{mean}\,[\exp(-s)\,\mathrm{BCE}(\mu,y)+\frac{1}{2}s]\ ;\quad$"
                   r"decide $\sigma(\mu_l)\geq\tau_l$", ha="center", fontsize=10)
    caption(ax, ymax, "Figure 4.  UCH internals: dual logit+uncertainty head with per-label threshold calibration.")
    save(fig, "fig4_uch")
fig1(); fig2(); fig3(); fig4()
print("architecture figures ->", HERE)


## 7) Results & paste-ready LaTeX

In [ ]:
# 7) Collect results — everything the paper needs
import json, glob
from IPython.display import Image, display
print("JSON files:")
for f in sorted(glob.glob(f"{OUT_DIR}/*.json")): print("  ", f)
print("\n==== PASTE-READY LaTeX (results/latex_tables.tex) ====\n")
print(open(f"{OUT_DIR}/latex_tables.tex").read())
for img in ["training_curves.png", "confusion_matrices.png", "attention_heatmap.png"]:
    p = f"{OUT_DIR}/{img}"
    if glob.glob(p):
        print(img); display(Image(p))
